# Semana 6 · El costo de no mirar los datos
**Fundamentos de Big Data — Taller práctico con Python y pandas**

> *Esta empresa tiene los datos. ¿Qué está dejando de ver?*

## El caso

**Olist** es un *marketplace* brasileño que conecta pequeñas tiendas con grandes plataformas de comercio electrónico. Entre 2016 y 2018 registró cerca de **100 000 pedidos**, con información de clientes, productos, pagos, entregas y reseñas.

Su equipo acaba de ser contratado como consultor. El gerente les dice:

> *"Tenemos años de registros guardados en el sistema, pero nunca los hemos analizado. No sé si sirven para algo."*

Su misión es demostrarle, **con cifras**, qué está ganando la empresa si usa sus datos y qué está perdiendo si los sigue ignorando.

## Objetivos de aprendizaje

- Cargar y explorar un conjunto de datos empresarial real con **pandas**.
- Responder preguntas de negocio combinando varias tablas.
- Identificar **beneficios** de aplicar el análisis de datos y **riesgos** de no hacerlo.


### Cómo trabajar con este notebook

1. Si lo abrió en Google Colab: **Archivo → Guardar una copia en Drive** antes de empezar.
2. Ejecute las celdas en orden con **Shift + Enter**.
3. Las celdas marcadas con **COMPLETE AQUÍ** tienen espacios `___` que usted debe reemplazar por el código correcto.
4. Las preguntas numeradas (**Pregunta 1.1**, **Pregunta 2.1**…) se responden en la celda de texto que aparece debajo (doble clic para editarla).


## 0. Preparar el entorno

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams["figure.figsize"] = (10, 4)

### Carga de los datos

Usamos el **Brazilian E-Commerce Public Dataset by Olist** ([Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)).

La celda siguiente busca los archivos en este orden:

1. **Carpeta `datos/`** junto al notebook (por ejemplo, si subió los CSV a Colab con el ícono de carpeta o si clonó el repositorio).
2. **Google Drive**, en `Mi unidad/olist/` (si el docente compartió la carpeta; primero monte Drive con el ícono de carpeta → *Montar Drive*).
3. **Descarga directa desde Kaggle** con `kagglehub`. Si le pide credenciales, use la opción 1 o 2.


In [ ]:
import os

def obtener_carpeta_datos():
    """Devuelve la carpeta donde están los CSV de Olist."""
    candidatas = ["datos", "../datos", "/content/datos", "/content/drive/MyDrive/olist"]
    for carpeta in candidatas:
        if os.path.exists(os.path.join(carpeta, "olist_orders_dataset.csv")):
            print("Datos encontrados en:", carpeta)
            return carpeta
    print("Descargando desde Kaggle (puede tardar un minuto)...")
    try:
        import kagglehub
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
    carpeta = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
    print("Datos descargados en:", carpeta)
    return carpeta

CARPETA = obtener_carpeta_datos()
sorted(f for f in os.listdir(CARPETA) if f.endswith(".csv"))

In [ ]:
fechas_pedido = ["order_purchase_timestamp", "order_approved_at",
                 "order_delivered_carrier_date", "order_delivered_customer_date",
                 "order_estimated_delivery_date"]

pedidos    = pd.read_csv(f"{CARPETA}/olist_orders_dataset.csv", parse_dates=fechas_pedido)
items      = pd.read_csv(f"{CARPETA}/olist_order_items_dataset.csv")
resenas    = pd.read_csv(f"{CARPETA}/olist_order_reviews_dataset.csv")
clientes   = pd.read_csv(f"{CARPETA}/olist_customers_dataset.csv")
productos  = pd.read_csv(f"{CARPETA}/olist_products_dataset.csv")
pagos      = pd.read_csv(f"{CARPETA}/olist_order_payments_dataset.csv")
categorias = pd.read_csv(f"{CARPETA}/product_category_name_translation.csv", encoding="utf-8-sig")
print("Tablas cargadas")

## 1. Primer vistazo: ¿qué datos tiene la empresa?

Antes de analizar, hay que saber qué hay. Cada tabla registra una parte distinta del negocio.

In [ ]:
tablas = {"pedidos": pedidos, "items": items, "resenas": resenas, "clientes": clientes,
          "productos": productos, "pagos": pagos}

resumen = pd.DataFrame({
    "filas":    {nombre: len(t) for nombre, t in tablas.items()},
    "columnas": {nombre: t.shape[1] for nombre, t in tablas.items()},
})
resumen

In [ ]:
pedidos.head()

In [ ]:
print("Primer pedido:", pedidos["order_purchase_timestamp"].min())
print("Último pedido:", pedidos["order_purchase_timestamp"].max())

**Pregunta 1.1.** ¿Qué tabla tiene más filas y por qué cree que es así? ¿Qué procesos del negocio quedan registrados en estas tablas?

*Respuesta:* 

## 2. Pregunta de negocio 1: ¿cuándo vende más la empresa?

Este primer análisis viene **resuelto como ejemplo**. Léalo con atención: los siguientes siguen la misma lógica.

Pasos: unir los ítems vendidos con la fecha del pedido (`merge`), extraer el mes y sumar el precio por mes (`groupby`).

In [ ]:
ventas = items.merge(pedidos, on="order_id")
ventas["mes"] = ventas["order_purchase_timestamp"].dt.to_period("M")

ventas_mes = ventas.groupby("mes")["price"].sum()

# Los meses de 2016 y los de septiembre-octubre de 2018 están incompletos en el dataset
ventas_mes = ventas_mes.loc["2017-01":"2018-08"]

ax = ventas_mes.plot(kind="line", marker="o", title="Ingresos por mes")
ax.set_ylabel("Ingresos (R$)")
ax.set_xlabel("")
plt.show()

print("Mes con más ventas:", ventas_mes.idxmax(), f"→ R$ {ventas_mes.max():,.0f}")

Ahora analice **a qué hora del día** compran los clientes.

**COMPLETE AQUÍ**: extraiga la hora de la fecha de compra. Pista: así como `.dt.to_period("M")` extrae el mes, existe un atributo `.dt.___` para la hora.

In [ ]:
# SOLUCIÓN
ventas["hora"] = ventas["order_purchase_timestamp"].dt.hour

pedidos_por_hora = ventas.groupby("hora")["order_id"].nunique()
pedidos_por_hora.plot(kind="bar", title="Pedidos según la hora de compra")
plt.show()

**Pregunta 2.1.** ¿En qué mes vendió más la empresa? ¿Qué evento comercial pudo causarlo? ¿A qué horas casi no hay pedidos?

**Pregunta 2.2.** ¿Qué decisiones concretas podría tomar el gerente con esta información (inventario, personal, publicidad)?

*Respuesta:* 

## 3. Pregunta de negocio 2: ¿qué categorías generan los ingresos?

Los nombres de categoría están en portugués; la tabla `categorias` trae la traducción al inglés.

In [ ]:
prod = (items.merge(productos, on="product_id")
             .merge(categorias, on="product_category_name", how="left"))
prod[["product_id", "product_category_name", "product_category_name_english", "price"]].head()

**COMPLETE AQUÍ**: agrupe por la categoría en inglés y **sume** el precio.

In [ ]:
# SOLUCIÓN
top10 = (prod.groupby("product_category_name_english")["price"].sum()
             .sort_values(ascending=False)
             .head(10))

top10.sort_values().plot(kind="barh", title="Top 10 categorías por ingresos")
plt.xlabel("Ingresos (R$)")
plt.show()

**Principio de Pareto.** ¿Es cierto que unas pocas categorías concentran la mayor parte de los ingresos?

In [ ]:
por_categoria = prod.groupby("product_category_name_english")["price"].sum().sort_values(ascending=False)
acumulado = por_categoria.cumsum() / por_categoria.sum()
n_80 = (acumulado < 0.80).sum() + 1

print(f"{n_80} de {len(por_categoria)} categorías generan el 80 % de los ingresos "
      f"({n_80 / len(por_categoria):.0%} de las categorías).")

**Pregunta 3.1.** ¿Qué haría con esta información un jefe de compras? ¿Y un jefe de mercadeo?

*Respuesta:* 

## 4. Pregunta de negocio 3: ¿las entregas tardías afectan la satisfacción?

Trabajamos solo con pedidos **entregados** que tienen fecha de entrega. Un pedido es *tardío* si la fecha real de entrega es posterior a la fecha estimada que se le prometió al cliente.

In [ ]:
entregados = pedidos[(pedidos["order_status"] == "delivered")
                     & pedidos["order_delivered_customer_date"].notna()].copy()
len(entregados)

**COMPLETE AQUÍ**: cree la columna `tarde` comparando la fecha **real** de entrega con la fecha **estimada**.

In [ ]:
# SOLUCIÓN
entregados["tarde"] = entregados["order_delivered_customer_date"] > entregados["order_estimated_delivery_date"]

print(f"Pedidos entregados tarde: {entregados['tarde'].mean():.1%}")

**COMPLETE AQUÍ**: una los pedidos entregados con las reseñas y calcule la calificación **promedio** (`mean`) y el número de pedidos (`count`) según si llegaron tarde o no.

In [ ]:
# SOLUCIÓN
df = entregados.merge(resenas, on="order_id")
df.groupby("tarde")["review_score"].agg(["mean", "count"])

Veamos con más detalle: ¿cambia la calificación según **cuántos días** se retrasa el pedido?

In [ ]:
df["dias_retraso"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days

df["rango"] = pd.cut(
    df["dias_retraso"],
    bins=[-1000, -7, 0, 3, 7, 14, 1000],
    labels=["Más de 7 días antes", "A tiempo (0-6 días antes)", "1-3 días tarde",
            "4-7 días tarde", "8-14 días tarde", "Más de 14 días tarde"],
)

calif_por_rango = df.groupby("rango", observed=True)["review_score"].mean()
ax = calif_por_rango.plot(kind="bar", title="Calificación promedio según la puntualidad de la entrega")
ax.set_ylabel("Calificación (1 a 5)")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.show()

**Pregunta 4.1.** ¿Cuánto baja la calificación cuando un pedido llega tarde? ¿Qué pasa a partir de una semana de retraso?

**Pregunta 4.2.** Si la empresa nunca cruzara estas dos tablas (logística y reseñas), ¿cómo explicaría que sus calificaciones bajan?

*Respuesta:* 

## 5. Pregunta de negocio 4: ¿los clientes vuelven a comprar?

**Detalle importante del dataset:** Olist asigna un `customer_id` **nuevo en cada pedido**. Para identificar a la misma persona hay que usar `customer_unique_id`. Entender cómo se generó un dato es parte del trabajo con datos.

In [ ]:
pc = pedidos.merge(clientes, on="customer_id")
print("customer_id distintos:       ", pc["customer_id"].nunique())
print("customer_unique_id distintos:", pc["customer_unique_id"].nunique())

**COMPLETE AQUÍ**: agrupe por el identificador **único** del cliente y **cuente** cuántos pedidos tiene cada uno.

In [ ]:
# SOLUCIÓN
compras = pc.groupby("customer_unique_id")["order_id"].count()

print(f"Clientes que compraron solo una vez: {(compras == 1).mean():.1%}")

In [ ]:
compras.clip(upper=5).value_counts().sort_index().rename(
    index={5: "5 o más"}).rename("clientes").to_frame()

**Pregunta 5.1.** ¿Qué significa para el negocio este porcentaje? Investigue: ¿cuánto más cuesta conseguir un cliente nuevo que conservar uno actual?

*Respuesta:* 

## 6. Pregunta de negocio 5 (opcional): ¿cómo pagan los clientes?

In [ ]:
pagos["payment_type"].value_counts(normalize=True).mul(100).round(1).rename("% de pagos").to_frame()

**COMPLETE AQUÍ**: calcule el número **promedio** de cuotas (`payment_installments`) por tipo de pago.

In [ ]:
# SOLUCIÓN
pagos.groupby("payment_type")["payment_installments"].mean().sort_values(ascending=False)

**Pregunta 6.1.** ¿Qué oportunidad o riesgo financiero ve en estos resultados?

*Respuesta:* 

## 7. Tablero de hallazgos

Ejecute esta celda para reunir las cifras clave que usted calculó. Si alguna falla, revise la sección correspondiente.

In [ ]:
print("HALLAZGOS CLAVE")
print("-" * 50)
print(f"Mes pico de ventas:              {ventas_mes.idxmax()}")
print(f"Categorías que generan el 80 %:  {n_80} de {len(por_categoria)}")
print(f"Pedidos entregados tarde:        {entregados['tarde'].mean():.1%}")
calif = df.groupby("tarde")["review_score"].mean()
print(f"Calificación a tiempo / tarde:   {calif.get(False, float('nan')):.2f} / {calif.get(True, float('nan')):.2f}")
print(f"Clientes que compran una vez:    {(compras == 1).mean():.1%}")

## 8. Del hallazgo a la decisión

Complete la tabla con **al menos cuatro** hallazgos. Cada uno debe llevar una **cifra** calculada en este notebook.

| # | Hallazgo (con cifra) | Área (clientes, operación, finanzas, estrategia) | Beneficio si la empresa lo usa | Riesgo si lo ignora |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |


### Pitch para el gerente

Escriba un mensaje de **máximo 150 palabras** (lo que diría en un minuto) que empiece así:

> *"Usted ya tiene un activo que no está usando…"*

*Pitch:* 

### Reflexión: puente a la semana 7

**Pregunta final.** Con estas tablas respondimos muchas preguntas, pero no todas. ¿Qué **otros datos** necesitaría la empresa para entender por qué los clientes no vuelven? ¿Dónde estarían esos datos (dentro o fuera de la empresa)?

*Respuesta:* 

---
## Entregable de la semana 6

- [ ] Notebook con todas las celdas COMPLETE AQUÍ completadas y ejecutadas sin errores.
- [ ] Preguntas respondidas.
- [ ] Tabla *Del hallazgo a la decisión* con al menos 4 filas.
- [ ] Pitch para el gerente.

Descárguelo con **Archivo → Descargar → Descargar .ipynb** y súbalo a la plataforma del curso.
